
# ARC-v0.17.1 — Multi-Random Allocation Audit

This notebook performs the final statistical closure audit for ARC-v0.17.

ARC-v0.17 established a **Risk-Aware Selective Fidelity (RASF)** policy on untouched FEVER validation queries. The original matched-random comparator used one deterministic budget-matched random allocation per operating point.

This audit asks a stricter question:

> **Across many independent matched-budget random allocations, how often does random selective SQ8 feedback match or exceed the observed RASF utility?**

No retrieval is rerun.

For each query \(q\), ARC-v0.17 already stores:

- \(U_q^{L}\): final nDCG@10 under Always-PQ32 feedback;
- \(U_q^{H}\): final nDCG@10 under Always-SQ8 feedback;
- \(U_q^{R}\): final nDCG@10 under the frozen RASF allocation.

For a uniformly random subset \(S\) of exactly \(m\) queries,

\[
U_{\mathrm{rand}}(S)
=
\frac{1}{N}
\left(
\sum_{q\in S}U_q^{H}
+
\sum_{q\notin S}U_q^{L}
\right).
\]

Equivalently,

\[
U_{\mathrm{rand}}(S)
=
\bar U^L+
\frac{1}{N}\sum_{q\in S}
\left(U_q^H-U_q^L\right).
\]

This lets us evaluate **10,000 random matched-budget allocations per configuration and budget without rerunning FAISS**.

## Primary test

For the primary 25% budget:

\[
H_0:
\text{RASF is no better than a uniformly random matched-budget allocation.}
\]

The Monte Carlo one-sided p-value is

\[
p=
\frac{
1+\#\{U_{\mathrm{rand}}\ge U_{\mathrm{RASF}}\}
}{
R+1
}.
\]

The `+1` correction prevents reporting a zero Monte Carlo p-value.

This is a post-confirmatory robustness audit and does not alter the sealed H1–H4 confirmation.


In [ ]:

# ============================================================
# Cell 1 — Imports / Drive / constants
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import warnings

import numpy as np
import pandas as pd

from google.colab import drive

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 20260816
RANDOM_REPS = 10_000
BUDGETS = [0.10, 0.25, 0.50]
PRIMARY_BUDGET = 0.25

DRIVE_ROOT = Path("/content/drive/MyDrive")

if not DRIVE_ROOT.is_dir():
    drive.mount("/content/drive")

assert DRIVE_ROOT.is_dir(), "Google Drive mount failed"

ARC_ROOT = (
    DRIVE_ROOT
    / "rag-pq-checkpoints"
    / "arc-v0"
)

assert ARC_ROOT.is_dir(), ARC_ROOT

def sha256_file(path, chunk_size=16 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

print("Drive:", DRIVE_ROOT)
print("ARC root:", ARC_ROOT)
print("Random allocations per cell:", RANDOM_REPS)


In [ ]:

# ============================================================
# Cell 2 — Locate the completed ARC-v0.17 run
# ============================================================

V017_ROOT = (
    ARC_ROOT
    / "fever-deployable-selective-fidelity-closure-v017"
)

assert V017_ROOT.is_dir(), (
    f"ARC-v0.17 root missing: {V017_ROOT}"
)

required_files = [
    "v017_policy_quality_summary.csv",
    "v017_paired_quality_contrasts.csv",
    "v017_closure_decision_table.csv",
    "v017_deployable_selective_fidelity_closure_report.json",
]

def is_complete_v017_run(path):
    if not path.is_dir():
        return False

    if not all(
        (path / name).is_file()
        for name in required_files
    ):
        return False

    checkpoints = list(
        path.glob("*.parquet")
    )

    # 2 configs × (2 always policies + 3 budgets × 2 selective policies)
    return len(checkpoints) >= 16

candidate_runs = sorted(
    [
        p
        for p in V017_ROOT.iterdir()
        if p.is_dir()
    ],
    reverse=True,
)

valid_runs = [
    p
    for p in candidate_runs
    if is_complete_v017_run(p)
]

print("Candidate v0.17 runs:", len(candidate_runs))

for p in candidate_runs:
    print(
        " ",
        p.name,
        "complete=",
        is_complete_v017_run(p),
    )

assert valid_runs, (
    "No complete ARC-v0.17 run found."
)

V017_RUN = valid_runs[0]

REPORT_PATH = (
    V017_RUN
    / "v017_deployable_selective_fidelity_closure_report.json"
)

report_v017 = json.loads(
    REPORT_PATH.read_text(
        encoding="utf-8"
    )
)

assert (
    report_v017["status"]
    ==
    "ARC_V017_DEPLOYABLE_SELECTIVE_FIDELITY_CLOSURE_COMPLETE"
)

assert report_v017["test_accessed"] is False

print()
print("Selected v0.17 run:", V017_RUN)
print("Claim gate:", report_v017["claim_gate"])
print("v0.17 report SHA:", sha256_file(REPORT_PATH))

assert report_v017["claim_gate"] == "STRONG_CLOSURE", (
    "This audit was designed for the completed STRONG_CLOSURE run."
)

print("ARC-v0.17 SOURCE INTEGRITY — PASS")


In [ ]:

# ============================================================
# Cell 3 — Load canonical per-query ARC-v0.17 checkpoints
# ============================================================

POLICY_CONFIGS = [
    "mean-k20-a0p3",
    "softmax-k5-a0p5-t0p1",
]

def find_checkpoint(config_name, policy, budget=None):
    if budget is None:
        name = f"{config_name}__{policy}.parquet"
    else:
        name = (
            f"{config_name}__{policy}"
            f"-b{int(round(100 * budget)):02d}.parquet"
        )

    p = V017_RUN / name
    assert p.is_file(), p
    return p

data = {}

for cfg in POLICY_CONFIGS:
    data[cfg] = {}

    for policy in [
        "always_pq32",
        "always_sq8_feedback",
    ]:
        p = find_checkpoint(
            cfg,
            policy,
        )

        df = pd.read_parquet(p)

        assert {
            "query_id",
            "final_ndcg10",
        }.issubset(df.columns)

        assert df["query_id"].nunique() == 3316

        data[cfg][policy] = (
            df[
                [
                    "query_id",
                    "final_ndcg10",
                ]
            ]
            .copy()
        )

    for budget in BUDGETS:
        for policy in [
            "risk_selective",
            "random_selective",
        ]:
            p = find_checkpoint(
                cfg,
                policy,
                budget,
            )

            df = pd.read_parquet(p)

            assert {
                "query_id",
                "final_ndcg10",
            }.issubset(df.columns)

            assert df["query_id"].nunique() == 3316

            data[cfg][
                (policy, budget)
            ] = (
                df[
                    [
                        "query_id",
                        "final_ndcg10",
                    ]
                ]
                .copy()
            )

print("Canonical per-query checkpoints loaded.")
print("QUERY-LEVEL SOURCE LOAD — PASS")


In [ ]:

# ============================================================
# Cell 4 — Verify selective-policy compositional identity
#
# Because ARC-v0.17 selects SQ8 feedback for an entire query
# trajectory, every risk/random selective final utility should
# equal either that query's Always-PQ32 or Always-SQ8 value.
# ============================================================

identity_rows = []

for cfg in POLICY_CONFIGS:
    base = (
        data[cfg]["always_pq32"]
        .rename(
            columns={
                "final_ndcg10":
                    "u_pq32",
            }
        )
    )

    upper = (
        data[cfg]["always_sq8_feedback"]
        .rename(
            columns={
                "final_ndcg10":
                    "u_sq8",
            }
        )
    )

    ref = base.merge(
        upper,
        on="query_id",
        how="inner",
        validate="one_to_one",
    )

    assert len(ref) == 3316

    for budget in BUDGETS:
        for policy in [
            "risk_selective",
            "random_selective",
        ]:
            sel = (
                data[cfg][(policy, budget)]
                .rename(
                    columns={
                        "final_ndcg10":
                            "u_selective",
                    }
                )
            )

            m = ref.merge(
                sel,
                on="query_id",
                how="inner",
                validate="one_to_one",
            )

            is_base = np.isclose(
                m["u_selective"],
                m["u_pq32"],
                atol=1e-12,
                rtol=0.0,
            )

            is_upper = np.isclose(
                m["u_selective"],
                m["u_sq8"],
                atol=1e-12,
                rtol=0.0,
            )

            consistent = (
                is_base
                | is_upper
            )

            identity_rows.append({
                "config_name":
                    cfg,

                "budget":
                    budget,

                "policy":
                    policy,

                "rows":
                    len(m),

                "fraction_equal_base_or_upper":
                    float(
                        consistent.mean()
                    ),

                "nonmatching_rows":
                    int(
                        (~consistent).sum()
                    ),
            })

identity_audit = pd.DataFrame(
    identity_rows
)

display(identity_audit)

assert (
    identity_audit[
        "nonmatching_rows"
    ]
    == 0
).all(), (
    "Selective checkpoint contains final utilities that are not "
    "identical to the corresponding Always-PQ32/Always-SQ8 outcome. "
    "Offline random-allocation reconstruction would not be valid."
)

print("SELECTIVE COMPOSITIONAL IDENTITY — PASS")


In [ ]:

# ============================================================
# Cell 4.5 — Create ARC-v0.17.1 output directory
# ============================================================

V0171_ROOT = (
    ARC_ROOT
    / "fever-multi-random-allocation-audit-v0171"
)

V0171_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

RUN_ID = (
    datetime.now(
        timezone.utc
    )
    .strftime(
        "%Y%m%d-%H%M%S"
    )
)

OUT = (
    V0171_ROOT
    / RUN_ID
)

OUT.mkdir(
    parents=True,
    exist_ok=False,
)

print("v0.17.1 output:", OUT)



## Random-allocation null distribution

For each configuration, define the per-query benefit of SQ8 feedback over PQ32 feedback:

\[
g_q = U_q^{H} - U_q^{L}.
\]

A matched-budget random allocation of exactly \(m\) SQ8-feedback queries has utility

\[
\bar U^L + \frac{1}{N}\sum_{q\in S_m}g_q.
\]

Therefore the null distribution depends only on the fixed vector \(\{g_q\}_{q=1}^N\), and can be evaluated without rerunning retrieval.


In [ ]:

# ============================================================
# Cell 5 — Monte Carlo matched-budget random-allocation audit
# ============================================================

audit_rows = []
null_distributions = {}

for cfg_idx, cfg in enumerate(POLICY_CONFIGS):
    base = (
        data[cfg]["always_pq32"]
        .rename(
            columns={
                "final_ndcg10":
                    "u_pq32",
            }
        )
    )

    upper = (
        data[cfg]["always_sq8_feedback"]
        .rename(
            columns={
                "final_ndcg10":
                    "u_sq8",
            }
        )
    )

    m = base.merge(
        upper,
        on="query_id",
        how="inner",
        validate="one_to_one",
    )

    N = len(m)
    assert N == 3316

    u_base = (
        m[
            "u_pq32"
        ]
        .to_numpy(
            np.float64
        )
    )

    gains = (
        m[
            "u_sq8"
        ]
        .to_numpy(
            np.float64
        )
        -
        u_base
    )

    base_mean = float(
        u_base.mean()
    )

    full_gain_mean = float(
        gains.mean()
    )

    for budget in BUDGETS:
        n_select = int(
            round(
                budget
                * N
            )
        )

        risk_df = (
            data[cfg][
                (
                    "risk_selective",
                    budget,
                )
            ]
        )

        rasf_utility = float(
            risk_df[
                "final_ndcg10"
            ].mean()
        )

        original_random_df = (
            data[cfg][
                (
                    "random_selective",
                    budget,
                )
            ]
        )

        original_random_utility = float(
            original_random_df[
                "final_ndcg10"
            ].mean()
        )

        # Exact expectation under a uniformly random subset
        # of fixed size n_select.
        exact_random_expectation = (
            base_mean
            +
            (
                n_select
                / N
            )
            * full_gain_mean
        )

        local_seed = (
            SEED
            + 100_000 * cfg_idx
            + int(
                round(
                    10_000
                    * budget
                )
            )
        )

        local_rng = np.random.default_rng(
            local_seed
        )

        null = np.empty(
            RANDOM_REPS,
            dtype=np.float64,
        )

        for rep in range(
            RANDOM_REPS
        ):
            selected = local_rng.choice(
                N,
                size=n_select,
                replace=False,
            )

            null[rep] = (
                base_mean
                +
                gains[
                    selected
                ].sum()
                / N
            )

        null_distributions[
            (
                cfg,
                budget,
            )
        ] = null

        exceed_count = int(
            (
                null
                >= rasf_utility
            ).sum()
        )

        # Corrected Monte Carlo p-value.
        p_one_sided = (
            exceed_count
            + 1
        ) / (
            RANDOM_REPS
            + 1
        )

        percentile = float(
            (
                null
                < rasf_utility
            ).mean()
        )

        audit_rows.append({
            "config_name":
                cfg,

            "budget":
                budget,

            "queries":
                N,

            "selected_queries":
                n_select,

            "rasf_final_ndcg10":
                rasf_utility,

            "original_single_random_final_ndcg10":
                original_random_utility,

            "random_exact_expectation":
                exact_random_expectation,

            "random_mc_mean":
                float(
                    null.mean()
                ),

            "random_mc_std":
                float(
                    null.std(
                        ddof=1
                    )
                ),

            "random_mc_q025":
                float(
                    np.quantile(
                        null,
                        0.025,
                    )
                ),

            "random_mc_q50":
                float(
                    np.quantile(
                        null,
                        0.50,
                    )
                ),

            "random_mc_q975":
                float(
                    np.quantile(
                        null,
                        0.975,
                    )
                ),

            "rasf_minus_random_expectation":
                (
                    rasf_utility
                    -
                    exact_random_expectation
                ),

            "rasf_percentile_vs_random":
                percentile,

            "random_allocations_ge_rasf":
                exceed_count,

            "monte_carlo_one_sided_p":
                p_one_sided,

            "random_reps":
                RANDOM_REPS,

            "seed":
                local_seed,
        })

allocation_audit = pd.DataFrame(
    audit_rows
)

display(
    allocation_audit
)

allocation_audit.to_csv(
    OUT
    / "v0171_multi_random_allocation_audit.csv",
    index=False,
)

print("MULTI-RANDOM ALLOCATION AUDIT — COMPLETE")


In [ ]:

# ============================================================
# Cell 6 — Compare the original single random allocation with
# the full random null distribution
# ============================================================

single_random_rows = []

for _, row in allocation_audit.iterrows():
    cfg = row["config_name"]
    budget = float(
        row["budget"]
    )

    null = null_distributions[
        (
            cfg,
            budget,
        )
    ]

    single_u = float(
        row[
            "original_single_random_final_ndcg10"
        ]
    )

    single_percentile = float(
        (
            null
            < single_u
        ).mean()
    )

    single_random_rows.append({
        "config_name":
            cfg,

        "budget":
            budget,

        "single_random_utility":
            single_u,

        "random_null_mean":
            float(
                null.mean()
            ),

        "single_random_percentile":
            single_percentile,

        "single_random_within_95pct_null_interval":
            bool(
                (
                    single_u
                    >= np.quantile(
                        null,
                        0.025,
                    )
                )
                and
                (
                    single_u
                    <= np.quantile(
                        null,
                        0.975,
                    )
                )
            ),
    })

single_random_audit = pd.DataFrame(
    single_random_rows
)

display(
    single_random_audit
)

single_random_audit.to_csv(
    OUT
    / "v0171_single_random_representativeness.csv",
    index=False,
)

print("SINGLE-RANDOM REPRESENTATIVENESS — COMPLETE")


In [ ]:

# ============================================================
# Cell 7 — Primary 25% claim gate
# ============================================================

primary = (
    allocation_audit[
        np.isclose(
            allocation_audit[
                "budget"
            ],
            PRIMARY_BUDGET,
        )
    ]
    .copy()
)

display(
    primary[
        [
            "config_name",
            "budget",
            "rasf_final_ndcg10",
            "random_exact_expectation",
            "random_mc_q025",
            "random_mc_q975",
            "rasf_minus_random_expectation",
            "rasf_percentile_vs_random",
            "random_allocations_ge_rasf",
            "monte_carlo_one_sided_p",
        ]
    ]
)

primary_pass_mask = (
    primary[
        "rasf_final_ndcg10"
    ]
    >
    primary[
        "random_mc_q975"
    ]
)

all_primary_pass = bool(
    primary_pass_mask.all()
)

any_primary_pass = bool(
    primary_pass_mask.any()
)

if all_primary_pass:
    CLAIM_GATE = (
        "STRONG_MULTI_RANDOM_CLOSURE"
    )
elif any_primary_pass:
    CLAIM_GATE = (
        "PARTIAL_MULTI_RANDOM_CLOSURE"
    )
else:
    CLAIM_GATE = (
        "NO_MULTI_RANDOM_CLOSURE"
    )

print()
print("=" * 80)
print("PRIMARY 25% MULTI-RANDOM DECISION")
print("=" * 80)

for _, row in primary.iterrows():
    print()
    print(
        row[
            "config_name"
        ]
    )
    print(
        "  RASF:",
        row[
            "rasf_final_ndcg10"
        ],
    )
    print(
        "  random expectation:",
        row[
            "random_exact_expectation"
        ],
    )
    print(
        "  random 95% upper:",
        row[
            "random_mc_q975"
        ],
    )
    print(
        "  allocations >= RASF:",
        int(
            row[
                "random_allocations_ge_rasf"
            ]
        ),
        "/",
        RANDOM_REPS,
    )
    print(
        "  corrected p:",
        row[
            "monte_carlo_one_sided_p"
        ],
    )

print()
print(
    "CLAIM GATE:",
    CLAIM_GATE,
)


In [ ]:

# ============================================================
# Cell 8 — Optional finite-population analytic check
#
# For a simple random sample without replacement of n_select
# gains from N gains:
#
# Var(sum selected gains)
#   = n * (1 - n/N) * S^2
#
# where S^2 uses denominator N-1.
# Utility divides the selected gain sum by N.
# ============================================================

analytic_rows = []

for cfg in POLICY_CONFIGS:
    base = (
        data[cfg]["always_pq32"]
        .rename(
            columns={
                "final_ndcg10":
                    "u_pq32",
            }
        )
    )

    upper = (
        data[cfg]["always_sq8_feedback"]
        .rename(
            columns={
                "final_ndcg10":
                    "u_sq8",
            }
        )
    )

    m = base.merge(
        upper,
        on="query_id",
        validate="one_to_one",
    )

    gains = (
        m["u_sq8"]
        -
        m["u_pq32"]
    ).to_numpy(
        np.float64
    )

    N = len(gains)
    S2 = float(
        np.var(
            gains,
            ddof=1,
        )
    )

    for budget in BUDGETS:
        n_select = int(
            round(
                budget
                * N
            )
        )

        variance_sum = (
            n_select
            *
            (
                1.0
                -
                n_select
                / N
            )
            *
            S2
        )

        analytic_sd_utility = (
            math.sqrt(
                variance_sum
            )
            / N
        )

        mc_row = allocation_audit[
            (
                allocation_audit[
                    "config_name"
                ]
                == cfg
            )
            &
            np.isclose(
                allocation_audit[
                    "budget"
                ],
                budget,
            )
        ].iloc[0]

        analytic_rows.append({
            "config_name":
                cfg,

            "budget":
                budget,

            "analytic_random_sd":
                analytic_sd_utility,

            "monte_carlo_random_sd":
                float(
                    mc_row[
                        "random_mc_std"
                    ]
                ),

            "absolute_sd_difference":
                abs(
                    analytic_sd_utility
                    -
                    float(
                        mc_row[
                            "random_mc_std"
                        ]
                    )
                ),
        })

analytic_check = pd.DataFrame(
    analytic_rows
)

display(
    analytic_check
)

analytic_check.to_csv(
    OUT
    / "v0171_random_variance_analytic_check.csv",
    index=False,
)

print("FINITE-POPULATION VARIANCE CHECK — COMPLETE")


In [ ]:

# ============================================================
# Cell 9 — Paper-facing summary
# ============================================================

paper_rows = []

for _, row in allocation_audit.iterrows():
    paper_rows.append({
        "config_name":
            row[
                "config_name"
            ],

        "budget":
            float(
                row[
                    "budget"
                ]
            ),

        "rasf_ndcg10":
            float(
                row[
                    "rasf_final_ndcg10"
                ]
            ),

        "random_expected_ndcg10":
            float(
                row[
                    "random_exact_expectation"
                ]
            ),

        "delta_rasf_vs_random_expectation":
            float(
                row[
                    "rasf_minus_random_expectation"
                ]
            ),

        "random_95pct_low":
            float(
                row[
                    "random_mc_q025"
                ]
            ),

        "random_95pct_high":
            float(
                row[
                    "random_mc_q975"
                ]
            ),

        "rasf_random_percentile":
            float(
                row[
                    "rasf_percentile_vs_random"
                ]
            ),

        "corrected_one_sided_p":
            float(
                row[
                    "monte_carlo_one_sided_p"
                ]
            ),
    })

paper_summary = pd.DataFrame(
    paper_rows
)

display(
    paper_summary
)

paper_summary.to_csv(
    OUT
    / "v0171_paper_facing_summary.csv",
    index=False,
)

print("PAPER-FACING SUMMARY — COMPLETE")


In [ ]:

# ============================================================
# Cell 10 — Seal ARC-v0.17.1 report
# ============================================================

report = {
    "status":
        "ARC_V0171_MULTI_RANDOM_ALLOCATION_AUDIT_COMPLETE",

    "source_v017_run":
        str(
            V017_RUN
        ),

    "source_v017_report_sha256":
        sha256_file(
            REPORT_PATH
        ),

    "random_reps_per_config_budget":
        RANDOM_REPS,

    "budgets":
        BUDGETS,

    "primary_budget":
        PRIMARY_BUDGET,

    "claim_gate":
        CLAIM_GATE,

    "all_primary_configs_pass":
        all_primary_pass,

    "any_primary_config_pass":
        any_primary_pass,

    "test_accessed":
        False,

    "statistical_definition": {
        "null":
            (
                "uniform random allocation without replacement "
                "of exactly the matched SQ8-feedback query count"
            ),

        "one_sided_p":
            (
                "(1 + count(random utility >= RASF utility)) "
                "/ (R + 1)"
            ),
    },

    "interpretation_constraints": [
        (
            "This audit quantifies random-allocation uncertainty "
            "conditional on the per-query Always-PQ32 and "
            "Always-SQ8-feedback outcomes from ARC-v0.17."
        ),
        (
            "It does not rerun retrieval and therefore introduces "
            "no additional retrieval stochasticity."
        ),
        (
            "The primary scientific unit remains the query; this "
            "allocation audit addresses uncertainty in the matched "
            "random allocation itself."
        ),
        (
            "This is post-confirmatory and does not alter the sealed "
            "H1-H4 results."
        ),
    ],

    "completed_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

FINAL_REPORT_PATH = (
    OUT
    / "v0171_multi_random_allocation_audit_report.json"
)

FINAL_REPORT_PATH.write_text(
    json.dumps(
        report,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

report_sha = sha256_file(
    FINAL_REPORT_PATH
)

(
    OUT
    / "V0171_REPORT_SHA256.txt"
).write_text(
    report_sha
    + "  "
    + FINAL_REPORT_PATH.name
    + "\n",
    encoding="utf-8",
)

print()
print("=" * 90)
print("ARC-v0.17.1 MULTI-RANDOM ALLOCATION AUDIT — COMPLETE")
print("=" * 90)
print("Claim gate:", CLAIM_GATE)
print("Output:", OUT)
print("Report SHA-256:", report_sha)
print("Test accessed:", False)
print("=" * 90)
